In [8]:
!pip install pandas 
!pip install prefect 


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached click-8.4.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached opentelemetry_api-1.42.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached orjson-3.11.9-cp314-cp314-win_amd64.whl.metadata (43 kB)
  Using cached pluggy-1.6.0-py3-none-any.whl.metadata (4.8 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached pydantic_settings-2.14.1-py3-none-any.whl.metadata (3.4 kB)
  Using cached pytz-2026.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-win_amd64.whl.metadata (2.4 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import datetime
import pandas as pd
from prefect import flow, task



In [10]:

@task(retries=3, retry_delay_seconds=10)
def extract_transactions() -> pd.DataFrame:
    """
    E: Extrae el lote de transacciones más reciente.
    Tiene configurados 3 reintentos automáticos por si la 'fuente' falla.
    """
    # Simulamos datos crudos con ruido: nulos, strings en montos y duplicados
    raw_data = {
        "id_transaccion": [1001, 1002, 1003, 1004, 1005, 1001],  # 1001 duplicado
        "id_cliente": ["C-201", "C-202", "C-203", "C-204", "C-205", "C-201"],
        "monto": ["$15,200.00", "$350.50", None, "$85,000.00", "$1,200.00", "$15,200.00"],
        "tipo_operacion": ["transferencia", "tarjeta", "retiro", "transferencia", "tarjeta", "transferencia"],
        "fecha": ["2026-06-15", "2026-06-16", "2026-06-16", "2026-06-17", "2026-06-17", "2026-06-15"]
    }
    df = pd.DataFrame(raw_data)
    print(f" Extracción exitosa. Registros crudos encontrados: {len(df)}")
    return df


@task
def transform_and_score(df: pd.DataFrame, threshold: float = 50000.0) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    T: Limpia los datos, normaliza los formatos y aplica la lógica de negocio
    para detectar transacciones de alto riesgo.
    """
    print(" Iniciando fase de transformación y análisis de riesgo...")
    
    # 1. Depuración: Eliminar duplicados exactos
    df = df.drop_duplicates()
    
    # 2. Limpieza: Formatear el monto a float y manejar valores nulos
    df["monto"] = df["monto"].str.replace("$", "", regex=False).str.replace(",", "", regex=False)
    df["monto"] = pd.to_numeric(df["monto"], errors="coerce")
    df["monto"] = df["monto"].fillna(0.0)  # Si es nulo, asumimos 0 para no romper el cálculo
    
    # 3. Normalización: Asegurar formato de fecha estándar
    df["fecha"] = pd.to_datetime(df["fecha"]).dt.strftime("%Y-%m-%d")
    
    # 4. Lógica de Negocio (Regla de Riesgo): 
    # Marcamos transacciones inusualmente altas o huérfanas (monto 0 pero con ID)
    df["alto_riesgo"] = (df["monto"] >= threshold) | (df["monto"] == 0.0)
    
    # Separamos el dataset en dos: Operaciones Normales y Alertas de Riesgo
    df_normal = df[df["alto_riesgo"] == False].copy()
    df_alerts = df[df["alto_riesgo"] == True].copy()
    
    return df_normal, df_alerts


@task
def load_to_storage(df_normal: pd.DataFrame, df_alerts: pd.DataFrame):
    """
    L: Carga los datos procesados en sus respectivos destinos.
    """
    print(" Iniciando fase de carga...")
    
    # Aquí simularíamos un .to_sql() a PostgreSQL o almacenamiento en un Data Lake
    total_normal = len(df_normal)
    total_alerts = len(df_alerts)
    
    print(f" Carga Finalizada:")
    print(f"   - {total_normal} transacciones normales enviadas al histórico comercial.")
    print(f"   - {total_alerts} alertas críticas enviadas al panel de cumplimiento.")
    
    if total_alerts > 0:
        print("\n⚠️ ¡ALERTA DE SEGURIDAD! Revisar los siguientes folios de inmediato:")
        print(df_alerts[["id_transaccion", "id_cliente", "monto", "tipo_operacion"]])


@flow(name="Pipeline-Riesgo-Financiero", log_prints=True)
def run_financial_etl():
    """
    El Flujo Principal (Orquestador).
    Define la secuencia exacta y dependencias de las tareas.
    """
    # Ejecución secuencial controlada por Prefect
    raw_df = extract_transactions()
    clean_df, alert_df = transform_and_score(raw_df, threshold=40000.0)
    load_to_storage(clean_df, alert_df)


if __name__ == "__main__":
    # Arrancamos el pipeline de manera local
    run_financial_etl()

18:33:45.172 | INFO    | prefect - Starting temporary server on http://127.0.0.1:8704
See https://docs.prefect.io/v3/concepts/server#how-to-guides for more information on running a dedicated Prefect server.

RuntimeError: Failed to reach API at http://127.0.0.1:8704/api/